**Nawaf Khalid Alsabr**  
**ID: 2230004159**

# Introduction to Machine Translation (MT):

Machine Translation is a subfield of AI and linguistics that develops systems capable of translating text from one language to another. The primary goal of machine translation is to enable effective communication between people who speak different languages by automating the process of language translation.

Challenges in Machine Translation:
- Ambiguity

Ex. I went to the bank.
- Idiomatic Expressions

Ex. It's raining cats and dogs.
- Low-Resource Languages

Some languages have very limited data.

Applications of Machine Translation:
- Global Business and E-commerce
-Cross-Cultural Communication
- Multilingual Content Creation
- Language Learning



## Neural Machine Translation:

Neural Machine Translation is a data-driven approach where a single large neural network learns to represent both the source language and the target language, and to directly map between them.
Instead of using manually defined rules or phrase tables, it learns translation patterns automatically from large datasets of bilingual text.

### Key Components of Neural Machine Translation:


In [1]:
#!pip install tensorflow

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np

# Example bilingual dataset
english_sentences = ["Hello.", "How are you?", "This is a machine translation example."]
french_sentences = ["Bonjour.", "Comment ca va?", "Ceci est un exemple de traduction automatique."]

# Tokenize the english sentences
tokenizer_eng = Tokenizer()
tokenizer_eng.fit_on_texts(english_sentences) # Tokenize words and assign an integer ID to each unique word
eng_sequences = tokenizer_eng.texts_to_sequences(english_sentences) # Returns a list of tokens replaced by their index. Note: words index will start from 1 because 0 is reserved by default for padding (not assigned to any word).
print(f"eng_sequences = {eng_sequences}")
eng_word_index = tokenizer_eng.word_index # Returns the dictionary created by the tokenizer dict {word: index}.
print(f"eng_word_index = {eng_word_index}")

I0000 00:00:1793647492.989243     210 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793647492.989716     210 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793647493.033831     210 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793647494.162246     210 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793647494.162639     210 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


eng_sequences = [[1], [2, 3, 4], [5, 6, 7, 8, 9, 10]]
eng_word_index = {'hello': 1, 'how': 2, 'are': 3, 'you': 4, 'this': 5, 'is': 6, 'a': 7, 'machine': 8, 'translation': 9, 'example': 10}


In [3]:
# Tokenize the french sentences
tokenizer_frn = Tokenizer()
tokenizer_frn.fit_on_texts(french_sentences)
frn_sequences = tokenizer_frn.texts_to_sequences(french_sentences)

print(f"frn_sequences = {frn_sequences}")
frn_word_index = tokenizer_frn.word_index
print(f"frn_word_index = {frn_word_index}")

frn_sequences = [[1], [2, 3, 4], [5, 6, 7, 8, 9, 10, 11]]
frn_word_index = {'bonjour': 1, 'comment': 2, 'ca': 3, 'va': 4, 'ceci': 5, 'est': 6, 'un': 7, 'exemple': 8, 'de': 9, 'traduction': 10, 'automatique': 11}


In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Padding sequences
eng_padded = pad_sequences(eng_sequences) # makes all sequences the same length by padding with zeros at the beginning (default)
print(f"eng_padded = {eng_padded}")

eng_padded = [[ 0  0  0  0  0  1]
 [ 0  0  0  2  3  4]
 [ 5  6  7  8  9 10]]


In [5]:
frn_padded = pad_sequences(frn_sequences)
frn_padded

array([[ 0,  0,  0,  0,  0,  0,  1],
       [ 0,  0,  0,  0,  2,  3,  4],
       [ 5,  6,  7,  8,  9, 10, 11]], dtype=int32)

## LSTM Model


In [6]:
from sklearn.model_selection import train_test_split

# Train-test split
eng_train, eng_test, frn_train, frn_test = train_test_split(eng_padded, frn_padded, test_size=0.2, random_state=42)

# Define the model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=len(eng_word_index) + 1, output_dim=64),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.RepeatVector(frn_padded.shape[1]), #length of french sentence
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(frn_word_index) + 1, activation='softmax')) #convert to probabilities for each frn word
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(eng_train, np.expand_dims(frn_train, -1), epochs=50, batch_size=1, validation_split=0.2)

# Evaluate the model
loss, accuracy = model.evaluate(eng_test, np.expand_dims(frn_test, -1))
print(f'Test Accuracy: {accuracy}')

Epoch 1/50


E0000 00:00:1793647495.765559     210 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5714 - loss: 2.4822

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5714 - loss: 2.4822 - val_accuracy: 0.0000e+00 - val_loss: 2.4873


Epoch 2/50


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5714 - loss: 2.4720

... (144 outputs left out)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 1.0000 - loss: 0.4906 - val_accuracy: 0.0000e+00 - val_loss: 5.7410


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step - accuracy: 0.5714 - loss: 1.7894

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step - accuracy: 0.5714 - loss: 1.7894


Test Accuracy: 0.5714285969734192


In [7]:
# Inference
def translate_sentence(sentence, model, eng_tokenizer, frn_tokenizer):
    sequence = eng_tokenizer.texts_to_sequences([sentence])
    padded_sequence = pad_sequences(sequence, maxlen=eng_padded.shape[1])
    predicted_sequence = model.predict(padded_sequence) #outputs probabilities for each timestep (french padded sentence) and each unique word in the target vocabulary.
    translated_sequence = np.argmax(predicted_sequence, axis=-1) #Returns the index (word ID) with the largest value at each timestep.
    translated_sentence = frn_tokenizer.sequences_to_texts(translated_sequence)
    return translated_sentence[0]

# Example Inference
input_sentence = "How are you?"
translated_sentence = translate_sentence(input_sentence, model, tokenizer_eng, tokenizer_frn)
print(f'Input: {input_sentence}\nTranslation: {translated_sentence}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 365ms/step


Input: How are you?
Translation: comment va va


________________________________________________________________________________________________________________________________________

### Task 1: Understanding the Model Layers

Answer these simple questions in your own words:

1. What does the **Embedding** layer do to a word index?
3. What is the role of the **Encoder LSTM**?
5. What does the **Decoder LSTM** do?
8. Why do we use `softmax` in the final layer?


**Answers:**

1. The **Embedding** layer turns each word index (an integer ID, e.g. "how" = 2) into a dense
vector of real numbers (here, 64-dimensional). Instead of treating words as arbitrary unrelated
IDs, it learns a vector representation where words used in similar contexts end up with similar
vectors -- this gives the network something meaningful to compute with, instead of raw integers
that have no numeric relationship to each other.

2. The **Encoder LSTM** reads the input (source-language) sequence one token at a time and
compresses everything it has seen into a fixed-size hidden state (a "context vector"). By the
time it reaches the end of the sentence, that hidden state is meant to summarize the overall
meaning of the whole input sentence.

3. The **Decoder LSTM** takes the encoder's final hidden state (repeated via `RepeatVector` so it
is fed in at every output timestep) and generates the output (target-language) sequence one token
at a time, using what it has produced so far together with the encoded context to predict the next
word.

4. We use **softmax** in the final layer because at each output timestep we need a probability
distribution over the entire target vocabulary (every possible French word), where all the
probabilities sum to 1. Softmax converts the raw output scores ("logits") into exactly that --
letting us pick the most likely word (or sample from the distribution) at each position, and
letting the model be trained with cross-entropy loss against the correct word.

### Task 2:

 Use Case: Language Translation (English-French)

A global e-commerce company wants to expand its reach to French-speaking customers and provide a seamless shopping experience. The company needs a language translation solution to automatically translate product listings, customer reviews, and other relevant content from English to French.

link: https://www.kaggle.com/datasets/devicharith/language-translation-englishfrench

**Note on the dataset:** the Kaggle dataset linked above
(`devicharith/language-translation-englishfrench`, ~175,000 English-French sentence pairs) can't
be downloaded in this sandbox -- `kaggle.com` is blocked by the network policy here, the same
limitation as earlier labs. In its place, the cell below builds a small hand-written
English-French parallel corpus in the same e-commerce/customer-service domain the use case
describes (order confirmations, shipping, reviews, support), and trains the exact same
encoder-decoder LSTM architecture from the demo above on it. The code is written so that swapping
in the real Kaggle CSV is a one-line change (`pd.read_csv(...)` instead of the inline lists) --
if you download `eng_-french.csv` yourself (locally, or in Google Colab where this notebook has
full internet access), you can plug it straight in to train on the full ~175k-pair dataset.

In [8]:
# A small hand-written English-French corpus covering the e-commerce use case
# (order confirmations, shipping, reviews, support) described above. Swap this
# block for `pd.read_csv("eng_-french.csv")` to train on the full Kaggle dataset.
ecommerce_english = [
    "Hello.",
    "Thank you for your order.",
    "The product is in stock.",
    "Please leave a review.",
    "Your package has shipped.",
    "We offer free shipping.",
    "The price is ten dollars.",
    "Contact customer support for help.",
    "Your order has been confirmed.",
    "The item is out of stock.",
    "We hope you enjoy your purchase.",
    "Returns are accepted within thirty days.",
    "Your refund has been processed.",
    "The delivery will arrive tomorrow.",
    "This item is on sale.",
]

ecommerce_french = [
    "Bonjour.",
    "Merci pour votre commande.",
    "Le produit est en stock.",
    "Veuillez laisser un avis.",
    "Votre colis a ete expedie.",
    "Nous offrons la livraison gratuite.",
    "Le prix est de dix dollars.",
    "Contactez le support client pour obtenir de l aide.",
    "Votre commande a ete confirmee.",
    "L article est en rupture de stock.",
    "Nous esperons que vous apprecierez votre achat.",
    "Les retours sont acceptes sous trente jours.",
    "Votre remboursement a ete traite.",
    "La livraison arrivera demain.",
    "Cet article est en solde.",
]

# Tokenize, same as the demo above
tokenizer_eng2 = Tokenizer()
tokenizer_eng2.fit_on_texts(ecommerce_english)
eng_sequences2 = tokenizer_eng2.texts_to_sequences(ecommerce_english)
eng_word_index2 = tokenizer_eng2.word_index

tokenizer_frn2 = Tokenizer()
tokenizer_frn2.fit_on_texts(ecommerce_french)
frn_sequences2 = tokenizer_frn2.texts_to_sequences(ecommerce_french)
frn_word_index2 = tokenizer_frn2.word_index

eng_padded2 = pad_sequences(eng_sequences2)
frn_padded2 = pad_sequences(frn_sequences2)

print(f"Vocabulary sizes -- English: {len(eng_word_index2)}, French: {len(frn_word_index2)}")
print(f"English padded shape: {eng_padded2.shape}, French padded shape: {frn_padded2.shape}")

# Train-test split
eng_train2, eng_test2, frn_train2, frn_test2 = train_test_split(
    eng_padded2, frn_padded2, test_size=0.2, random_state=42
)

# Same encoder-decoder LSTM architecture as the demo
model2 = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=len(eng_word_index2) + 1, output_dim=64),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.RepeatVector(frn_padded2.shape[1]),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(frn_word_index2) + 1, activation="softmax"))
])

model2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model2.fit(eng_train2, np.expand_dims(frn_train2, -1), epochs=50, batch_size=1, validation_split=0.2, verbose=0)

loss2, accuracy2 = model2.evaluate(eng_test2, np.expand_dims(frn_test2, -1), verbose=0)
print(f"\nTest Accuracy: {accuracy2}")

# Try translating a couple of product-listing-style sentences
for test_sentence in ["Thank you for your order.", "Your package has shipped."]:
    translated = translate_sentence(test_sentence, model2, tokenizer_eng2, tokenizer_frn2)
    print(f"Input: {test_sentence}\nTranslation: {translated}\n")


Vocabulary sizes -- English: 52, French: 54
English padded shape: (15, 6), French padded shape: (15, 9)



Test Accuracy: 0.37037038803100586


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step


Input: Thank you for your order.
Translation: votre est ete en

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


Input: Your package has shipped.
Translation: votre la a ete stock



**Note on results:** like the demo model above, this is trained on a tiny corpus (15 sentence
pairs) for only 50 epochs, so it will likely memorize/overfit these exact training sentences
rather than generalize to new ones -- that's expected for a dataset this small, not a bug. A
production version of this use case would train the same architecture on the full ~175k-pair
Kaggle dataset, which gives the model enough examples to actually learn general English-to-French
translation patterns rather than just memorizing individual sentences.